# Preparação do Dataset CIFAR-10 e Divisão de Validação

Este notebook realiza a preparação do dataset CIFAR-10 para os experimentos de treinamento e poda subsequentes. O objetivo principal é obter uma partição de dados consistente e reprodutível através das seguintes etapas:

1. **Download do CIFAR-10:** Carregamento dos conjuntos oficiais de treino e teste do Torchvision.
2. **Visualização de Classes:** Exibição de amostras visuais de cada uma das 10 classes do dataset.
3. **Divisão Estratificada Determinística (Train-Validation Split):** Isolamento de 15% das imagens de treino originais para validação (garantindo 750 amostras por classe de forma determinística).
4. **Exportação dos Índices:** Salvamento dos índices da partição em disco (`data/cifar10_split.pt`) para garantir que todos os modelos utilizem a mesma base de comparação.

## 1. Configuração do Ambiente e Inicialização

Importação das bibliotecas do PyTorch, Torchvision, Scikit-learn, Matplotlib e configuração das sementes aleatórias (seed = 42) de forma determinística para assegurar a reprodutibilidade de todo o pipeline de dados.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import v2
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from collections import defaultdict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

cuda


## 2. Download do Dataset CIFAR-10

Baixa e carrega o conjunto oficial de treinamento e teste do dataset CIFAR-10 diretamente a partir do Torchvision.

In [ ]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

## 3. Visualização e Verificação de Amostras das Classes

Para verificar se os dados foram carregados corretamente, filtramos e exibimos uma amostra visual (imagem 32x32) para cada uma das 10 classes presentes no CIFAR-10.

In [ ]:
class_samples = {}
for img, label in trainset:
    if label not in class_samples:
        class_samples[label] = img
    if len(class_samples) == len(classes):
        break

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, (label, img) in enumerate(sorted(class_samples.items())):
    ax = axes[idx // 5, idx % 5]
    ax.imshow(img)
    ax.set_title(classes[label])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Divisão Estratificada Determinística (Treino/Validação)

Para treinar nossos modelos e avaliar a perda de desempenho pós-poda em um conjunto de validação idêntico, realizamos um split estratificado de **15%** (750 imagens por classe) a partir das 50.000 imagens de treino originais.

As sementes do gerador pseudo-aleatório do NumPy são configuradas individualmente por classe para tornar a divisão reprodutível.

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616))])

full_trainset  = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

targets = np.array(full_trainset.targets)
perc_val = 15
val_per_class = perc_val*50
train_indices = []
val_indices = []

for class_id in range(10):
    class_indices = np.where(targets == class_id)[0]

    # deterministic shuffle
    rng = np.random.default_rng(SEED + class_id)
    rng.shuffle(class_indices)

    val_class_indices = class_indices[:val_per_class]
    train_class_indices = class_indices[val_per_class:]

    val_indices.extend(val_class_indices)
    train_indices.extend(train_class_indices)


trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

batch_size = 64

valloader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
val_targets = targets[val_indices]

for i in range(10):
    print(f"Class {i}: {(val_targets == i).sum()}")

## 5. Exportação e Cache dos Índices da Partição

Salvamos o dicionário contendo os índices gerados no arquivo `data/cifar10_split.pt` para que outros notebooks ou scripts de treino possam recarregar exatamente a mesma divisão de treino e validação.

In [ ]:
train_indices = np.array(train_indices)
val_indices = np.array(val_indices)

torch.save(
    {
        "train_indices": train_indices,
        "val_indices": val_indices,
        "seed": SEED
    },
    "data/cifar10_split.pt"
)
print("Split saved.")